# TP2 NLP - Executor Colab

Este notebook apenas orquestra comandos do repositório. A implementação fica em `src`, `custom_metrics` e `scripts`.

## 1. Clonar ou entrar no repositório

Esta célula usa o repositório da disciplina. Como ele é privado, adicione um Secret no Colab chamado `GH_TOKEN` ou `GITHUB_TOKEN` com um token do GitHub com acesso ao repositório.


In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/ed-icomp-ufam/trabalho-pr-tico-2-nlp-2026-01-mh131105.git'
REPO_DIR = Path('/content/trabalho-pr-tico-2-nlp-2026-01-mh131105')
BRANCH = 'master'

def github_token():
    token = os.environ.get('GH_TOKEN') or os.environ.get('GITHUB_TOKEN')
    try:
        from google.colab import userdata
        token = token or userdata.get('GH_TOKEN') or userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    return token

token = github_token()
auth_url = REPO_URL
if token:
    auth_url = REPO_URL.replace('https://', f'https://{token}@')
else:
    print('Sem GH_TOKEN/GITHUB_TOKEN. Se o clone falhar, configure o token nos Secrets do Colab.')

if not (REPO_DIR / '.git').exists():
    !git clone --branch {BRANCH} {auth_url} {REPO_DIR}
else:
    print(f'Repositório já existe em {REPO_DIR}')

%cd {REPO_DIR}
!git status --short --branch


In [ ]:
import os

token = github_token()
if token:
    pull_url = REPO_URL.replace('https://', f'https://{token}@')
else:
    pull_url = 'origin'

!git pull --ff-only {pull_url} {BRANCH}


## 2. Instalar dependências

In [ ]:
import os

token = github_token()
if token:
    pull_url = REPO_URL.replace('https://', f'https://{token}@')
else:
    pull_url = 'origin'

!git pull --ff-only {pull_url} {BRANCH}
!grep torch requirements.txt || echo "torch nao esta no requirements"
!pip install -r requirements.txt


## 3. Login opcional no Hugging Face

Defina `HF_TOKEN` nos segredos do Colab ou ajuste a célula de login.

In [ ]:
from huggingface_hub import login
import os
login(token=os.environ.get('HF_TOKEN'))

## 4. Preparar datasets

O `prepare_spider` primeiro usa `data/raw/spider` se a pasta já existir. Se ela não existir, o script pode importar um diretório ou ZIP/TAR passado com `--source_path`, ou tentar usar a fonte configurada no Hugging Face.

In [ ]:
!python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider
# If the automatic source fails, put a Spider ZIP/folder on Drive and use:
# !python -m scripts.prepare_spider --data_dir data/raw/spider --output_dir data/processed/spider --source_path /content/drive/MyDrive/spider.zip --force_download
!python -m scripts.prepare_mmlu --config configs/eval.yaml

## 5. Benchmark do baseline

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base

In [ ]:
!python -m scripts.evaluate_mmlu --config configs/eval.yaml --model_path outputs/base --output_dir outputs/base

## 6. Treinar experimentos LoRA

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_a.yaml

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_b.yaml

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_c.yaml

In [ ]:
!python -m scripts.train --config configs/train_lora_exp_d.yaml

## 7. Benchmarks finais

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_a

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_b

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_c

In [ ]:
!python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/exp_d

## 8. Smoke mode sem baixar modelos

Use isto apenas para validar o encadeamento dos scripts, não para reportar resultados.

In [ ]:
# !python -m pytest
# !python -m scripts.prepare_mmlu --config configs/eval.yaml --mock --limit_per_category 2
# !python -m scripts.run_benchmarks --config configs/eval.yaml --model_path outputs/base --mock --limit 2